<a href="https://colab.research.google.com/github/ljvillarrealm/SMAZ-2026/blob/main/HW_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Question 1: [Index] S&P 500 Stocks Added to the Index

**Which year had the highest number of additions (starting from 2020)?**

Using the list of S&P 500 companies from Wikipedia's [S&P 500 companies page](https://en.wikipedia.org/wiki/List_of_S%26P_500_companies), download the data including the year each company was added to the index.

Hint: you can use [pandas.read_html](https://pandas.pydata.org/docs/reference/api/pandas.read_html.html) to scrape the data into a DataFrame. You may need to add headers to make it work:
```python
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

# Fetch the HTML content with headers
response = requests.get(url, headers=headers)
```

Steps:
1. Create a DataFrame with company tickers, names, and the year they were added.
2. Extract the year from the addition date and calculate the number of stocks added each year.
3. Which (full) year had the highest number of additions, starting from 2020?

▎ Note: When stocks are added to the S&P 500, they usually experience a price bump as investors and index funds buy     
  shares following the announcement.

▎ Context:
> "Following the announcement, all four new entrants saw their stock prices rise in extended trading on Friday" - recent examples of S&P 500 additions include DASH, WSM, EXE, TKO in 2025 ([Nasdaq article](https://www.nasdaq.com/articles/sp-500-reshuffle-dash-tko-expe-wsm-join-worth-buying)).

*Additional*: How many current S&P 500 stocks have been in the index for more than 20 years?

In [1]:
import requests
import pandas as pd
from datetime import datetime

# 1. Scrape the S&P 500 companies list from Wikipedia
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

# Fetch the HTML content with headers
response = requests.get(url, headers=headers)

# Use pandas to read HTML tables
tables = pd.read_html(response.text)

# The S&P 500 companies are usually in the first table
sp500_df = tables[0]

# Rename columns for clarity if needed, or select relevant ones
# The column for addition date might have variations, so let's inspect and standardize

# Expected columns: 'Security', 'Symbol', 'Date first added'
# Let's check the actual column names from the scraped DataFrame
# print(sp500_df.columns)

# Assuming 'Date first added' is the correct column name after inspecting the table
# If the column name is slightly different, it needs to be adjusted here.
# For example, if it's 'Date added', use sp500_df['Date added'].

# A common name is 'Date first added' or 'Date Added'
# Let's try to find it dynamically or assume 'Date first added'

# Standardize column names for easier access
sp500_df.columns = [
    'Security', 'Symbol', 'GICS Sector', 'GICS Sub-Industry', 'Location',
    'Date first added', 'CIK', 'Founded'
]

# Convert 'Date first added' to datetime and extract the year
sp500_df['Date first added'] = pd.to_datetime(sp500_df['Date first added'], errors='coerce')
sp500_df['Addition Year'] = sp500_df['Date first added'].dt.year

# 2. Extract the year from the addition date and calculate the number of stocks added each year.
annual_additions = sp500_df.groupby('Addition Year').size().reset_index(name='Num Additions')

# 3. Which (full) year had the highest number of additions, starting from 2020?
annual_additions_since_2020 = annual_additions[annual_additions['Addition Year'] >= 2020]

if not annual_additions_since_2020.empty:
    max_additions_year = annual_additions_since_2020.loc[annual_additions_since_2020['Num Additions'].idxmax()]
    print(f"Year with the highest number of additions (starting from 2020): {int(max_additions_year['Addition Year'])} with {max_additions_year['Num Additions']} additions.")
else:
    print("No additions recorded since 2020 in the dataset.")

# Additional: How many current S&P 500 stocks have been in the index for more than 20 years?
current_year = datetime.now().year
year_20_years_ago = current_year - 20

stocks_over_20_years = sp500_df[sp500_df['Addition Year'] <= year_20_years_ago]
num_stocks_over_20_years = len(stocks_over_20_years)

print(f"Number of current S&P 500 stocks in the index for more than 20 years: {num_stocks_over_20_years}")

/tmp/ipykernel_1820/3176547656.py:13: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


Year with the highest number of additions (starting from 2020): 2025 with 18 additions.
Number of current S&P 500 stocks in the index for more than 20 years: 227


In [2]:
sp500_df

,Security,Symbol,GICS Sector,GICS Sub-Industry,Location,Date first added,CIK,Founded,Addition Year
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902,1957
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916,2017
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888,1957
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888),2012
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989,2011
...,...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969,2019
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927,2001


### Question 2. [Macro] Indexes YTD (as of 21 August 2026)

**How many indexes (out of 10) have better year-to-date returns than the US (S&P 500) as of August 21, 2026?**

Using Yahoo Finance World Indices data, compare the year-to-date (YTD) performance (1 January-21 August 2026, use the closing price to calculate the growth) of major stock market indexes for the following countries:
* United States - S&P 500 (^GSPC)
* China - Shanghai Composite (000001.SS)
* Hong Kong - HANG SENG INDEX (^HSI)
* Australia - S&P/ASX 200 (^AXJO)
* India - Nifty 50 (^NSEI)
* Canada - S&P/TSX Composite (^GSPTSE)
* Germany - DAX (^GDAXI)
* United Kingdom - FTSE 100 (^FTSE)
* Japan - Nikkei 225 (^N225)
* Mexico - IPC Mexico (^MXX)
* Brazil - Ibovespa (^BVSP)

*Hint*: use ```start_date='2026-01-01'``` and ```end_date='2026-08-21'``` parameters when downloading daily data in yfinance

Context:
> [Global Valuations: Who's Cheap, Who's Not?](https://simplywall.st/article/beyond-the-us-global-markets-after-yet-another-tariff-update) article suggests "Other regions may be growing faster than the US and you need to diversify."

Reference: Yahoo Finance World Indices - https://finance.yahoo.com/world-indices/

*Additional*: How many of these indexes have better returns than the S&P 500 over 3, 5, and 10 year periods? Do you see the same trend?
Note: For simplicity, ignore currency conversion effects.

In [2]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

# Define the tickers for the specified indices
index_tickers = {
    'United States - S&P 500': '^GSPC',
    'China - Shanghai Composite': '000001.SS',
    'Hong Kong - HANG SENG INDEX': '^HSI',
    'Australia - S&P/ASX 200': '^AXJO',
    'India - Nifty 50': '^NSEI',
    'Canada - S&P/TSX Composite': '^GSPTSE',
    'Germany - DAX': '^GDAXI',
    'United Kingdom - FTSE 100': '^FTSE',
    'Japan - Nikkei 225': '^N225',
    'Mexico - IPC Mexico': '^MXX',
    'Brazil - Ibovespa': '^BVSP'
}

# --- Part 1: YTD returns (1 January - 21 August 2026) ---
start_date_ytd = '2026-01-01'
end_date_ytd = '2026-08-21'

ytd_returns = {}

print(f"\n--- YTD Returns ({start_date_ytd} to {end_date_ytd}) ---")
for name, ticker in index_tickers.items():
    try:
        data = yf.download(ticker, start=start_date_ytd, end=end_date_ytd, progress=False, auto_adjust=True) # Added auto_adjust to suppress warning
        if not data.empty and len(data) > 1:
            start_price = data['Close'].iloc[0].item() # Extract scalar value
            end_price = data['Close'].iloc[-1].item()   # Extract scalar value
            ytd_return = (end_price / start_price - 1) * 100
            ytd_returns[name] = ytd_return
            print(f"{name} ({ticker}): {ytd_return:.2f}%")
        else:
            print(f"Could not retrieve sufficient data for {name} ({ticker}) for YTD calculation.")
            ytd_returns[name] = None
    except Exception as e:
        print(f"Error processing {name} ({ticker}) for YTD: {e}")
        ytd_returns[name] = None

snp_500_ytd_return = ytd_returns.get('United States - S&P 500')

if snp_500_ytd_return is not None:
    better_than_sp500_ytd = sum(1 for name, ret in ytd_returns.items() if ret is not None and ret > snp_500_ytd_return and name != 'United States - S&P 500')
    print(f"\nNumber of indexes with better YTD returns than the S&P 500: {better_than_sp500_ytd} out of {len(index_tickers) - 1}")
else:
    print("Cannot compare YTD returns as S&P 500 YTD data is not available.")

# --- Part 2: Returns over 3, 5, and 10 year periods ---
periods = {
    '3-year': 3,
    '5-year': 5,
    '10-year': 10
}

current_date = datetime(2026, 8, 21)

for period_name, years in periods.items():
    print(f"\n--- {period_name.capitalize()} Returns ---")
    period_start_date = (current_date - timedelta(days=years * 365)).strftime('%Y-%m-%d') # Approximation
    period_end_date = current_date.strftime('%Y-%m-%d')

    period_returns = {}
    for name, ticker in index_tickers.items():
        try:
            data = yf.download(ticker, start=period_start_date, end=period_end_date, progress=False, auto_adjust=True) # Added auto_adjust to suppress warning
            if not data.empty and len(data) > 1:
                start_price = data['Close'].iloc[0].item() # Extract scalar value
                end_price = data['Close'].iloc[-1].item()   # Extract scalar value
                returns = (end_price / start_price - 1) * 100
                period_returns[name] = returns
                print(f"{name} ({ticker}): {returns:.2f}%")
            else:
                print(f"Could not retrieve sufficient data for {name} ({ticker}) for {period_name} calculation.")
                period_returns[name] = None
        except Exception as e:
            print(f"Error processing {name} ({ticker}) for {period_name}: {e}")
            period_returns[name] = None

    snp_500_period_return = period_returns.get('United States - S&P 500')

    if snp_500_period_return is not None:
        better_than_sp500_period = sum(1 for name, ret in period_returns.items() if ret is not None and ret > snp_500_period_return and name != 'United States - S&P 500')
        print(f"Number of indexes with better {period_name} returns than the S&P 500: {better_than_sp500_period} out of {len(index_tickers) - 1}")
    else:
        print(f"Cannot compare {period_name} returns as S&P 500 data is not available.")


--- YTD Returns (2026-01-01 to 2026-08-21) ---
United States - S&P 500 (^GSPC): 11.41%
China - Shanghai Composite (000001.SS): -2.97%
Hong Kong - HANG SENG INDEX (^HSI): -2.43%
Australia - S&P/ASX 200 (^AXJO): 4.08%
India - Nifty 50 (^NSEI): -7.32%
Canada - S&P/TSX Composite (^GSPTSE): 14.06%
Germany - DAX (^GDAXI): 5.88%
United Kingdom - FTSE 100 (^FTSE): 8.01%
Japan - Nikkei 225 (^N225): 27.75%
Mexico - IPC Mexico (^MXX): 0.32%
Brazil - Ibovespa (^BVSP): 4.60%

Number of indexes with better YTD returns than the S&P 500: 2 out of 10

--- 3-year Returns ---
United States - S&P 500 (^GSPC): 74.16%
China - Shanghai Composite (000001.SS): 25.11%
Hong Kong - HANG SENG INDEX (^HSI): 44.45%
Australia - S&P/ASX 200 (^AXJO): 27.55%
India - Nifty 50 (^NSEI): 24.93%
Canada - S&P/TSX Composite (^GSPTSE): 84.68%
Germany - DAX (^GDAXI): 65.44%
United Kingdom - FTSE 100 (^FTSE): 47.83%
Japan - Nikkei 225 (^N225): 107.86%
Mexico - IPC Mexico (^MXX): 21.12%
Brazil - Ibovespa (^BVSP): 44.57%
Number of

### Question 3. [Index] S&P 500 Market Corrections Analysis


**Calculate the median drawdown (in %) of significant market corrections in the S&P 500 index.**

For this task, define a correction as an event when a stock index goes down by **at least 5%** from the most recent all-time high.

Steps:
1. Download S&P 500 historical data (period from 1950 to present) using yfinance (daily stats)
2. Identify all-time high closing price points (where price exceeds all previous days' prices)
3. For each pair of consecutive all-time highs, find the minimum price in between
4. Calculate drawdown percentages: (high - low) / high × 100
5. Filter for corrections with at least 5% drawdown
6. Calculate the duration in days for each correction period
7. Determine the 25th, 50th (median), and 75th percentiles for correction durations and drawdowns

*Context:*
> * Investors often wonder about the typical length of market corrections when deciding "when to buy the dip" ([Reddit discussion](https://www.reddit.com/r/investing/comments/1jrqnte/when_are_you_buying_the_dip/?rdt=64135)).

> * [A Wealth of Common Sense - How Often Should You Expect a Stock Market Correction?](https://awealthofcommonsense.com/2022/01/how-often-should-you-expect-a-stock-market-correction/)

*Hint (use this data to compare with your results)*: Here is the list of top 10 largest corrections by drawdown:
* 2007-10-09 to 2009-03-09: 56.8% drawdown over 517 days
* 2000-03-24 to 2002-10-09: 49.1% drawdown over 929 days
* 1973-01-11 to 1974-10-03: 48.2% drawdown over 630 days
* 1968-11-29 to 1970-05-26: 36.1% drawdown over 543 days
* 2020-02-19 to 2020-03-23: 33.9% drawdown over 33 days
* 1987-08-25 to 1987-12-04: 33.5% drawdown over 101 days
* 1961-12-12 to 1962-06-26: 28.0% drawdown over 196 days
* 1980-11-28 to 1982-08-12: 27.1% drawdown over 622 days
* 2022-01-03 to 2022-10-12: 25.4% drawdown over 282 days
* 1966-02-09 to 1966-10-07: 22.2% drawdown over 240 days

In [5]:
import yfinance as yf
import pandas as pd
from datetime import datetime

# 1. Download S&P 500 historical data (period from 1950 to present)
ticker_symbol = '^GSPC' # S&P 500 ticker
sp500_data = yf.download(ticker_symbol, start='1950-01-01', end=datetime.now().strftime('%Y-%m-%d'), progress=False)

if sp500_data.empty:
    print(f"Could not download data for {ticker_symbol}. Please check the ticker and internet connection.")
else:
    # Use 'Close' price for calculations
    close_prices = sp500_data['Close']

    # 2. Identify all-time high closing price points
    all_time_highs = close_prices.expanding().max()

    corrections = []
    current_high_idx = 0

    for i in range(1, len(close_prices)):
        # If a new all-time high is reached
        # Extract scalar values from Series for comparison
        if close_prices.iloc[i].item() > all_time_highs.iloc[i-1].item():
            if i > current_high_idx + 1: # Ensure there was a period after the last high
                # Period between current_high_idx and i-1 is the correction window
                correction_period_data = close_prices.iloc[current_high_idx : i]
                peak = correction_period_data.iloc[0].item() # The previous all-time high, extracted as scalar
                trough = correction_period_data.min().item() # The minimum price in the correction, extracted as scalar
                trough_date = correction_period_data.idxmin().item() # Date of the trough, extracted as scalar Timestamp

                drawdown = ((peak - trough) / peak) * 100

                # Calculate duration from peak to trough
                peak_date = close_prices.index[current_high_idx]
                duration = (trough_date - peak_date).days

                if drawdown >= 5: # Filter for corrections with at least 5% drawdown
                    corrections.append({
                        'Peak Date': peak_date,
                        'Trough Date': trough_date,
                        'Peak Price': peak,
                        'Trough Price': trough,
                        'Drawdown (%)': drawdown,
                        'Duration (days)': duration
                    })
            current_high_idx = i

    corrections_df = pd.DataFrame(corrections)

    if not corrections_df.empty:
        # Ensure 'Duration (days)' is numeric for percentile calculation
        corrections_df['Duration (days)'] = pd.to_numeric(corrections_df['Duration (days)'])

        # 7. Determine the 25th, 50th (median), and 75th percentiles
        # For Drawdown
        drawdown_25 = corrections_df['Drawdown (%)'].quantile(0.25)
        drawdown_50 = corrections_df['Drawdown (%)'].quantile(0.50)
        drawdown_75 = corrections_df['Drawdown (%)'].quantile(0.75)

        # For Duration
        duration_25 = corrections_df['Duration (days)'].quantile(0.25)
        duration_50 = corrections_df['Duration (days)'].quantile(0.50)
        duration_75 = corrections_df['Duration (days)'].quantile(0.75)

        print("\n--- S&P 500 Market Corrections Analysis (5% or greater drawdown) ---")
        print(f"Number of significant corrections found: {len(corrections_df)}")
        print("\nDrawdown Percentiles (%):")
        print(f"  25th Percentile: {drawdown_25:.2f}%")
        print(f"  Median (50th Percentile): {drawdown_50:.2f}%")
        print(f"  75th Percentile: {drawdown_75:.2f}%")

        print("\nDuration Percentiles (days):")
        print(f"  25th Percentile: {int(duration_25)} days")
        print(f"  Median (50th Percentile): {int(duration_50)} days")
        print(f"  75th Percentile: {int(duration_75)} days")

        # Optionally, display the corrections dataframe
        # print("\nDetails of significant corrections:")
        # print(corrections_df.round(2))
    else:
        print("No significant corrections (5% or greater drawdown) found in the downloaded data.")

/tmp/ipykernel_2204/517572703.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  sp500_data = yf.download(ticker_symbol, start='1950-01-01', end=datetime.now().strftime('%Y-%m-%d'), progress=False)



--- S&P 500 Market Corrections Analysis (5% or greater drawdown) ---
Number of significant corrections found: 74

Drawdown Percentiles (%):
  25th Percentile: 6.23%
  Median (50th Percentile): 7.99%
  75th Percentile: 14.02%

Duration Percentiles (days):
  25th Percentile: 22 days
  Median (50th Percentile): 40 days
  75th Percentile: 86 days


### Question 4.  [Stocks] Earnings Surprise Analysis for Amazon (AMZN)


**Calculate the median 2-day percentage change in stock prices following positive earnings surprise days.**

Steps:
1. Load earnings data for the last few years using the yfinance method get_earnings_dates():
```python
import yfinance as yf

ticker = 'AMZN'
ticker_obj = yf.Ticker(ticker)
result = ticker_obj.get_earnings_dates()
```
You should have 25 entries starting from 2020-10-29. Note: the 1 future earnings date entry will not have Reported EPS and Surprise % filled in.

2. Download complete historical price data using yfinance
3. Calculate 2-day percentage changes for all historical dates: for each sequence of 3 consecutive trading days (Day 1, Day 2, Day 3), compute the *return* as Close_Day3 / Close_Day1 - 1. (Assume Day 2 may correspond to the earnings announcement.)
4. Filter for positive earnings surprises and calculate the median 2-day return. Then calculate the correlation between
   the 2-day stock return and the earnings surprise magnitude.

▎ *Hint*: you can generate all correlations at once using pd.corr().

 > **Context:** Earnings announcements, especially when they exceed analyst expectations, can significantly impact stock prices in the short term.

**Reference:** [Yahoo Finance Earnings Calendar](https://finance.yahoo.com/calendar/earnings?symbol=AMZN)

*Additional*: Is there a correlation between the magnitude of the earnings surprise and the stock price reaction? Does the market react differently to earnings surprises during bull vs. bear markets?

In [15]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

ticker = 'AMZN'
ticker_obj = yf.Ticker(ticker)

# 1. Load earnings data for the last few years
# Fetch earnings dates, filtering for actual reported earnings
earnings_dates_raw = ticker_obj.earnings_dates

# Filter for entries that have 'Surprise(%)' (i.e., not future dates)
earnings_df = earnings_dates_raw.dropna(subset=['Surprise(%)']).copy()

# Ensure 'Reported Date' is datetime and set as index
earnings_df['Reported Date'] = pd.to_datetime(earnings_df.index)

# Convert 'Surprise(%)' to numeric, handling potential non-numeric values gracefully
earnings_df['Surprise(%)'] = pd.to_numeric(earnings_df['Surprise(%)'], errors='coerce')

# Filter for earnings from 2020 onwards as per prompt context
earnings_df = earnings_df[earnings_df['Reported Date'] >= '2020-01-01']

print(f"Loaded {len(earnings_df)} earnings entries for {ticker} since 2020.")

# 2. Download complete historical price data using yfinance
# Download data up to today for comprehensive 2-day return calculation
current_date = datetime.now().strftime('%Y-%m-%d')
historical_prices = yf.download(ticker, start='2019-01-01', end=current_date, progress=False, auto_adjust=True)

if historical_prices.empty:
    print("Could not download historical prices.")
else:
    # Normalize columns: yfinance can sometimes return MultiIndex columns, flatten them.
    # For a single ticker, the columns are typically ('Close', 'AMZN'), etc. We want the first level.
    if isinstance(historical_prices.columns, pd.MultiIndex):
        historical_prices.columns = historical_prices.columns.get_level_values(0)

    # Ensure 'Close' column name exists. With auto_adjust=True, 'Close' should already contain adjusted prices.
    if 'Adj Close' in historical_prices.columns and 'Close' not in historical_prices.columns:
        historical_prices = historical_prices.rename(columns={'Adj Close': 'Close'})
    elif 'Close' not in historical_prices.columns:
        print("Critical Error: 'Close' column not found in historical data after processing.")
        # Depending on criticality, you might want to raise an exception here.

    # 3. Calculate 2-day percentage changes
    # For each sequence of 3 consecutive trading days (Day 1, Day 2, Day 3),
    # compute the return as Close_Day3 / Close_Day1 - 1.
    # Assume Day 2 corresponds to the earnings announcement.
    # So, we want the return from close of Day1 to close of Day3.
    historical_prices['2_day_return'] = (historical_prices['Close'].shift(-2) / historical_prices['Close']) - 1

    # Align earnings dates with the historical prices for Day 2 (earnings announcement day)
    # The 'Reported Date' in earnings_df is the earnings announcement date (Day 2).
    # We need to map this to the 'Close' price on Day 1 from historical_prices
    # We can use the index of historical_prices, which are the dates.
    # We need to find the trading day *before* the reported date for Day 1's close.

    # Merge historical prices with earnings data on date
    # Need to handle cases where earnings date is not a trading day or first trading day after announcement.
    # Let's align earnings_df 'Reported Date' to historical_prices index.
    # We can use a forward fill to find the closest trading day for the reported date if it's not exact

    # Create a DataFrame to hold earnings and corresponding price data
    merged_data = pd.DataFrame(index=historical_prices.index)
    merged_data['Close'] = historical_prices['Close']
    merged_data['2_day_return'] = historical_prices['2_day_return']

    # Add earnings surprise to the merged data, aligning by date
    # We need to map the earnings surprise to the day *before* the 2-day return is calculated
    # If 'Reported Date' is Day 2, then '2_day_return' for that day uses Day 1's close.
    # So we map earnings_df.index (Reported Date) to the historical_prices index (which is Day 2).

    # Create a temporary series with earnings surprise indexed by reported date
    surprise_series = earnings_df.set_index('Reported Date')['Surprise(%)']

    # Reindex historical_prices to include all dates to correctly map earnings surprises
    all_dates = pd.date_range(start=historical_prices.index.min(), end=historical_prices.index.max())
    full_price_data = historical_prices.reindex(all_dates)

    # Forward fill the 'Close' and '2_day_return' for non-trading days
    # Then fill earnings surprise on the reported date
    full_price_data['Surprise(%)'] = surprise_series
    # Drop any rows where 'Close' is NaN (non-trading days that couldn't be forward filled from previous trading day)
    full_price_data.dropna(subset=['Close'], inplace=True)

    # Now, align the 'Surprise(%)' with the '2_day_return'. The 2_day_return is calculated from Close[t] to Close[t+2].
    # If earnings are announced on Day 2, then the 'Surprise(%)' should align with Day 2's return.
    # In our definition, `historical_prices['2_day_return']` at index `t` represents (Close[t+2] / Close[t]) - 1.
    # The earnings announcement happens *after* the close of Day 1 and *before* Day 2's open.
    # So, we should align the earnings surprise to the Day 2. The `Reported Date` is Day 2.
    # The '2_day_return' that reflects the reaction to this surprise would be from Day 2's close to Day 4's close.
    # However, the problem statement says "Close_Day3 / Close_Day1 - 1" where "Day 2 may correspond to the earnings announcement."
    # This implies the return is calculated *across* the announcement, i.e., from the day before the announcement to the day after.
    # So, if reported date is Day 2, we want return from Day 1 Close to Day 3 Close.
    # The `historical_prices['2_day_return']` at index `t` is `(Close[t+2] / Close[t]) - 1`.
    # If earnings are announced on `reported_date`, then `reported_date` is Day 2.
    # So we need the `2_day_return` corresponding to Day 1, which is `reported_date - 1 day`.

    # Let's create a DataFrame with the necessary columns for analysis
    analysis_df = pd.DataFrame(index=earnings_df['Reported Date'])
    analysis_df['Surprise(%)'] = earnings_df['Surprise(%)']

    # Find the trading day *before* the reported date (Day 1)
    # And the trading day *of* the reported date (Day 2 - if it's a trading day)
    # And the trading day *after* the reported date (Day 3)

    # Get a list of all trading dates, ensuring they are timezone-naive for consistency
    trading_days = historical_prices.index.normalize()

    # Create a mapping for reported dates to actual trading dates (Day -1 and Day +1)
    event_data = []
    for reported_date_tz, surprise_pct in earnings_df[['Surprise(%)']].itertuples():
        # Convert reported_date to timezone-naive for comparison with trading_days
        reported_date = reported_date_tz.tz_localize(None) if reported_date_tz.tzinfo is not None else reported_date_tz

        # Find the actual trading day 'D_minus_1' (Day 1)
        day1_target = reported_date - timedelta(days=1)
        day1_idx_arr = trading_days.get_indexer([day1_target], method='pad')
        day1_idx = day1_idx_arr[0] if len(day1_idx_arr) > 0 else -1

        # Find the actual trading day 'D_plus_1' (Day 3)
        day3_target = reported_date + timedelta(days=1)
        day3_idx_arr = trading_days.get_indexer([day3_target], method='bfill')
        day3_idx = day3_idx_arr[0] if len(day3_idx_arr) > 0 else -1

        if day1_idx != -1 and day3_idx != -1:
            day1_date = trading_days[day1_idx]
            day3_date = trading_days[day3_idx]

            close_day1 = historical_prices.loc[day1_date, 'Close'].item()
            close_day3 = historical_prices.loc[day3_date, 'Close'].item()

            if pd.notna(close_day1) and pd.notna(close_day3) and close_day1 != 0:
                two_day_return = (close_day3 / close_day1) - 1
                event_data.append({
                    'Reported Date': reported_date_tz, # Keep original timezone-aware date for record
                    'Surprise(%)': surprise_pct,
                    'Day1_Close': close_day1,
                    'Day3_Close': close_day3,
                    '2_Day_Return': two_day_return * 100 # Convert to percentage
                })

    event_df = pd.DataFrame(event_data)

    if not event_df.empty:
        # 4. Filter for positive earnings surprises and calculate the median 2-day return.
        positive_surprise_events = event_df[event_df['Surprise(%)'] > 0].copy()
        median_2day_return_positive_surprise = positive_surprise_events['2_Day_Return'].median()

        print(f"\nMedian 2-day percentage change following positive earnings surprise: {median_2day_return_positive_surprise:.2f}%")

        # Calculate the correlation between the 2-day stock return and the earnings surprise magnitude.
        correlation_all = event_df['Surprise(%)'].corr(event_df['2_Day_Return'])
        print(f"Correlation between 2-day return and earnings surprise magnitude (all events): {correlation_all:.2f}")

        if not positive_surprise_events.empty:
            correlation_positive = positive_surprise_events['Surprise(%)'].corr(positive_surprise_events['2_Day_Return'])
            print(f"Correlation between 2-day return and earnings surprise magnitude (positive surprises only): {correlation_positive:.2f}")
        else:
            print("No positive surprise events to calculate correlation for.")

        # Additional: Does the market react differently to earnings surprises during bull vs. bear markets?
        # To determine bull/bear, we'll use a simple long-term SMA of the S&P 500
        sp500_ticker = '^GSPC'
        sp500_data = yf.download(sp500_ticker, start='2019-01-01', end=current_date, progress=False, auto_adjust=True)
        sp500_data['SMA_200'] = sp500_data['Close'].rolling(window=200).mean()

        # Merge S&P 500 market condition with event_df based on the reported date
        # We need to align the market condition at the time of the earnings announcement (Day 2).
        # So we use `Reported Date` from `event_df` to lookup S&P 500 data.
        market_conditions = []
        for idx, row in event_df.iterrows():
            reported_date_tz = row['Reported Date']
            # Convert reported_date to timezone-naive for comparison with sp500_data.index
            reported_date = reported_date_tz.tz_localize(None) if reported_date_tz.tzinfo is not None else reported_date_tz

            # Find the closest S&P 500 close price and SMA_200 on or before the reported date
            market_date_idx_arr = sp500_data.index.get_indexer([reported_date], method='ffill')
            market_date_idx = market_date_idx_arr[0] if len(market_date_idx_arr) > 0 else -1

            if market_date_idx != -1:
                market_date = sp500_data.index[market_date_idx]
                sp500_close = sp500_data.loc[market_date, 'Close'].item() # Extract scalar
                sp500_sma = sp500_data.loc[market_date, 'SMA_200'].item() # Extract scalar

                if pd.notna(sp500_close) and pd.notna(sp500_sma):
                    market_state = 'Bull' if sp500_close > sp500_sma else 'Bear'
                else:
                    market_state = 'Unknown'
            else:
                market_state = 'Unknown'
            market_conditions.append(market_state)

        event_df['Market_State'] = market_conditions

        print("\n--- Market State Analysis ---")
        bull_market_events = event_df[event_df['Market_State'] == 'Bull']
        bear_market_events = event_df[event_df['Market_State'] == 'Bear']

        if not bull_market_events.empty:
            median_return_bull = bull_market_events[bull_market_events['Surprise(%)'] > 0]['2_Day_Return'].median()
            correlation_bull = bull_market_events['Surprise(%)'].corr(bull_market_events['2_Day_Return'])
            print(f"Bull Market: Median 2-day return (positive surprise): {median_return_bull:.2f}%, Correlation (all events): {correlation_bull:.2f}")
        else:
            print("No bull market events found.")

        if not bear_market_events.empty:
            median_return_bear = bear_market_events[bear_market_events['Surprise(%)'] > 0]['2_Day_Return'].median() # Corrected filter for positive surprises
            correlation_bear = bear_market_events['Surprise(%)'].corr(bear_market_events['2_Day_Return'])
            print(f"Bear Market: Median 2-day return (positive surprise): {median_return_bear:.2f}%, Correlation (all events): {correlation_bear:.2f}")
        else:
            print("No bear market events found.")

    else:
        print("No valid earnings events found for analysis.")

Loaded 24 earnings entries for AMZN since 2020.

Median 2-day percentage change following positive earnings surprise: 0.35%
Correlation between 2-day return and earnings surprise magnitude (all events): 0.22
Correlation between 2-day return and earnings surprise magnitude (positive surprises only): 0.31

--- Market State Analysis ---
Bull Market: Median 2-day return (positive surprise): -0.36%, Correlation (all events): 0.34
Bear Market: Median 2-day return (positive surprise): 1.05%, Correlation (all events): -0.17


### Question 5.  [Exploratory, optional] Brainstorm potential idea for your capstone project

**Free text answer**

Describe the capstone project you would like to pursue, considering your aspirations, ML model predictions, and prior knowledge. Even if you are unsure at this stage, try to generate an idea you would like to explore-such as a specific asset class, country, industry vertical, or investment strategy. Be as specific as possible.

*Example: I want to build a short-term prediction model for the US/India/Brazil stock markets, focusing on the largest stocks over a 30-day investment horizon. I plan to use RSI and MACD technical indicators and news coverage data to generate predictions.*

Create a poc of an automated trading system.

### Question 6. [Exploratory, optional] Investigate new metrics

**Free text answer**

Using the data sources we have covered (or any others you find relevant), download and explore a few additional metrics or time series that could be valuable for your project. Briefly explain why you think each metric is useful. This does not need to be a comprehensive list-focus on demonstrating your ability to generate data requests based on your project description, identify and locate the necessary data, and explain how you would retrieve it using Python.

GPR  at  https://www.matteoiacoviello.com/gpr.htm

## Submitting the solutions

Form for submitting: https://courses.datatalks.club/sma-zoomcamp-2026/homework/hw01

## Leaderboard

Leaderboard link: https://courses.datatalks.club/sma-zoomcamp-2026/leaderboard